# SemEval-2026 Task 13 — Ensemble Pipeline

**Soft Voting & Weighted Averaging** across CodeBERT, GraphCodeBERT, and UniXcoder

### Models
| Model | HF Hub | Pre-training Signal |
|:---|:---|:---|
| CodeBERT | `microsoft/codebert-base` | Bimodal (NL + PL) |
| GraphCodeBERT | `microsoft/graphcodebert-base` | + Data Flow Graph |
| UniXcoder | `microsoft/unixcoder-base` | + AST + Code Comments |

### Ensemble Strategies
1. **Soft Voting** — equal-weight probability averaging
2. **Weighted Averaging** — per-model weights (tuned on validation set)
3. **Rank Averaging** — rank-based fusion (robust to miscalibration)

### Setup
- **GPU:** Kaggle Tesla T4 / P100  
- **Metric:** Macro F1 (competition metric)

In [1]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
!pip install --upgrade transformers==4.45.0 huggingface_hub datasets scikit-learn accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 97.8 MB/s eta 0:00:00


In [2]:
# ============================================================
# Cell 2: Imports
# ============================================================
import os
os.environ["WANDB_DISABLED"] = "true"

import gc
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as nnF
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    f1_score,
)
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

2026-04-05 18:23:43.125633: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775413423.287457      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775413423.334731      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775413423.697757      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775413423.697790      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775413423.697793      24 computation_placer.cc:177] computation placer alr

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Configuration

In [3]:
# ============================================================
# Cell 3: Configuration
# ============================================================

# --- Task ---
TASK = "A"  # Change to "B" for multi-class authorship detection

# --- Models ---
MODELS = {
    "codebert":      "microsoft/codebert-base",
    "graphcodebert": "microsoft/graphcodebert-base",
    "unixcoder":     "microsoft/unixcoder-base",
}

# --- Data paths (Kaggle or HuggingFace fallback) ---
import os
from datasets import load_dataset
DATA_BASE = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13"

if not os.path.exists(DATA_BASE):
    print("Kaggle dataset not found. Downloading from HuggingFace...")
    hf_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", TASK)
    TRAIN_PATH = f"task_{TASK.lower()}_train.parquet"
    VAL_PATH = f"task_{TASK.lower()}_val.parquet"
    TEST_PATH = f"task_{TASK.lower()}_test.parquet"
    if not os.path.exists(TRAIN_PATH):
        hf_dataset['train'].to_parquet(TRAIN_PATH)
        hf_dataset['validation'].to_parquet(VAL_PATH)
        if 'test' in hf_dataset:
            hf_dataset['test'].to_parquet(TEST_PATH)
        else:
            TEST_PATH = None
    NUM_LABELS = 2 if TASK == "A" else 11
else:
    if TASK == "A":
        TRAIN_PATH = f"{DATA_BASE}/task_a/task_a_training_set_1.parquet"
        VAL_PATH   = f"{DATA_BASE}/task_a/task_a_validation_set.parquet"
        TEST_PATH  = f"{DATA_BASE}/task_a/task_a_test_set_sample.parquet"
        NUM_LABELS = 2
    elif TASK == "B":
        TRAIN_PATH = f"{DATA_BASE}/task_b/task_b_training_set_1.parquet"
        VAL_PATH   = f"{DATA_BASE}/task_b/task_b_validation_set.parquet"
        TEST_PATH  = f"{DATA_BASE}/task_b/task_b_test_set_sample.parquet"
        NUM_LABELS = 11

# --- Subsampling ---
TRAIN_SAMPLE_SIZE = 10000
VAL_SAMPLE_SIZE   = 2500
RANDOM_SEED       = 42

# --- Training hyperparameters ---
MAX_LENGTH       = 256
BATCH_SIZE       = 16
GRAD_ACCUM_STEPS = 2
LEARNING_RATE    = 2e-5
NUM_EPOCHS       = 5
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.1

# --- Ensemble ---
ENSEMBLE_STRATEGY = "weighted_avg"   # 'soft_vote', 'weighted_avg', 'rank_avg'
ENSEMBLE_WEIGHTS  = None  # None = optimise on val, or set e.g. [0.4, 0.35, 0.25]

# --- Output ---
OUTPUT_BASE   = "/kaggle/working"
ENSEMBLE_DIR  = f"{OUTPUT_BASE}/ensemble_cache"

print(f"Task: {TASK} ({NUM_LABELS} classes)")
print(f"Models: {list(MODELS.keys())}")
print(f"Strategy: {ENSEMBLE_STRATEGY}")

Kaggle dataset not found. Downloading from HuggingFace...


README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

task_a/task_a_training_set_1.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

task_a/task_a_validation_set.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

task_a/task_a_test_set_sample.parquet:   0%|          | 0.00/593k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Task: A (2 classes)
Models: ['codebert', 'graphcodebert', 'unixcoder']
Strategy: weighted_avg


## Data Loading

In [4]:
# ============================================================
# Cell 4: Data Loading + Stratified Subsampling
# ============================================================

def load_and_subsample(path, sample_size=None, seed=42):
    """Load parquet and optionally stratify-subsample."""
    df = pd.read_parquet(path)
    df = df.dropna(subset=['code', 'label'])
    df['label'] = df['label'].astype(int)
    
    if sample_size is not None and sample_size < len(df):
        df = df.groupby('label', group_keys=False).apply(
            lambda x: x.sample(
                n=max(1, int(sample_size * len(x) / len(df))),
                random_state=seed,
            )
        ).reset_index(drop=True)
    
    print(f"  Loaded: {len(df)} samples, {df['label'].nunique()} classes")
    print(f"  Distribution: {df['label'].value_counts().sort_index().to_dict()}")
    return df

print("Loading training data...")
train_df = load_and_subsample(TRAIN_PATH, TRAIN_SAMPLE_SIZE, RANDOM_SEED)

print("\nLoading validation data...")
val_df = load_and_subsample(VAL_PATH, VAL_SAMPLE_SIZE, RANDOM_SEED)

Loading training data...
  Loaded: 9999 samples, 2 classes
  Distribution: {0: 4769, 1: 5230}

Loading validation data...
  Loaded: 2499 samples, 2 classes
  Distribution: {0: 1192, 1: 1307}


## Individual Model Training

Train each model separately, then combine predictions.

In [5]:
# ============================================================
# Cell 5: Unified Trainer for all 3 models
# ============================================================

class UnifiedTrainer:
    """Single trainer class for CodeBERT / GraphCodeBERT / UniXcoder."""
    
    def __init__(self, model_name, tag, num_labels, max_length=MAX_LENGTH):
        self.model_name = model_name
        self.tag = tag
        self.num_labels = num_labels
        self.max_length = max_length
        self.tokenizer = None
        self.model = None
    
    def initialize(self):
        print(f"[{self.tag}] Initialising {self.model_name} ...")
        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)
        self.model = RobertaForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification",
        ).to('cuda')
        total = sum(p.numel() for p in self.model.parameters())
        print(f"[{self.tag}] {self.num_labels} labels | params {total:,}")
    
    def tokenize_fn(self, examples):
        return self.tokenizer(
            examples['code'], truncation=True, padding=True,
            max_length=self.max_length, return_tensors="pt",
        )
    
    def prepare_datasets(self, train_df, val_df):
        train_ds = Dataset.from_pandas(train_df[['code', 'label']])
        val_ds   = Dataset.from_pandas(val_df[['code', 'label']])
        
        train_ds = train_ds.map(self.tokenize_fn, batched=True, remove_columns=['code'])
        val_ds   = val_ds.map(self.tokenize_fn,   batched=True, remove_columns=['code'])
        
        train_ds = train_ds.rename_column('label', 'labels')
        val_ds   = val_ds.rename_column('label', 'labels')
        return train_ds, val_ds
    
    def compute_metrics(self, eval_pred):
        preds = np.argmax(eval_pred.predictions, axis=1)
        labels = eval_pred.label_ids
        return {
            'accuracy':  accuracy_score(labels, preds),
            'macro_f1':  f1_score(labels, preds, average='macro', zero_division=0),
            'f1':        f1_score(labels, preds, average='weighted', zero_division=0),
        }
    
    def train(self, train_ds, val_ds, output_dir, num_epochs=NUM_EPOCHS,
              batch_size=BATCH_SIZE, lr=LEARNING_RATE):
        print(f"[{self.tag}] Training ...")
        
        steps_per_epoch = len(train_ds) // (batch_size * GRAD_ACCUM_STEPS)
        eval_steps = max(200, steps_per_epoch // 2)
        
        args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            weight_decay=WEIGHT_DECAY,
            warmup_ratio=WARMUP_RATIO,
            logging_steps=50,
            eval_strategy="steps",
            eval_steps=eval_steps,
            save_strategy="steps",
            save_steps=eval_steps,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            learning_rate=lr,
            lr_scheduler_type="cosine",
            save_total_limit=2,
            report_to=[],
            fp16=True,
            remove_unused_columns=False,
        )
        
        hf_trainer = Trainer(
            model=self.model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=self.tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer=self.tokenizer),
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )
        
        hf_trainer.train()
        hf_trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        print(f"[{self.tag}] Model saved → {output_dir}")
        return hf_trainer

print("UnifiedTrainer defined.")

UnifiedTrainer defined.


In [6]:
# ============================================================
# Cell 6: Train all 3 models sequentially
# ============================================================

trained_models = {}  # tag → UnifiedTrainer instance

for tag, model_name in MODELS.items():
    print("\n" + "=" * 70)
    print(f"  TRAINING: {tag} ({model_name})")
    print("=" * 70)
    
    trainer_obj = UnifiedTrainer(
        model_name=model_name,
        tag=tag,
        num_labels=NUM_LABELS,
        max_length=MAX_LENGTH,
    )
    trainer_obj.initialize()
    
    train_ds, val_ds = trainer_obj.prepare_datasets(train_df, val_df)
    out_dir = f"{OUTPUT_BASE}/results_{tag}_task{TASK}"
    
    hf_trainer = trainer_obj.train(train_ds, val_ds, output_dir=out_dir)
    
    # Quick validation
    preds = hf_trainer.predict(val_ds)
    y_pred = np.argmax(preds.predictions, axis=1)
    val_f1 = f1_score(preds.label_ids, y_pred, average='macro', zero_division=0)
    print(f"[{tag}] Val Macro F1: {val_f1:.4f}")
    
    trained_models[tag] = trainer_obj
    
    # Free trainer memory (keep model for ensemble)
    del hf_trainer
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ All {len(trained_models)} models trained.")


  TRAINING: codebert (microsoft/codebert-base)
[codebert] Initialising microsoft/codebert-base ...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

Map:   0%|          | 0/2499 [00:00<?, ? examples/s]

[unixcoder] Training ...


Step,Training Loss,Validation Loss,Accuracy,Macro F1,F1
200,0.059500,0.101061,0.971589,0.971469,0.971554
400,0.031700,0.131729,0.975190,0.975161,0.975200
600,0.055700,0.206731,0.979992,0.979962,0.979998


[unixcoder] Model saved → /kaggle/working/results_unixcoder_taskA


[unixcoder] Val Macro F1: 0.9800

✅ All 3 models trained.


---
## Ensemble Inference

Combine predictions from all three models using soft voting or weighted averaging.

In [7]:
# ============================================================
# Cell 7: Ensemble — Per-Model Inference on Test Set
# ============================================================

@torch.no_grad()
def get_model_probs(trainer_obj, codes, batch_size=32, device="cuda"):
    """
    Run inference and return softmax probabilities.
    Returns: np.ndarray of shape (N, num_labels)
    """
    model = trainer_obj.model
    tokenizer = trainer_obj.tokenizer
    model.to(device)
    model.eval()
    
    all_probs = []
    for i in tqdm(range(0, len(codes), batch_size), desc=f"  [{trainer_obj.tag}]"):
        batch = codes[i:i + batch_size]
        enc = tokenizer(
            batch, truncation=True, padding=True,
            max_length=MAX_LENGTH, return_tensors="pt",
        )
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        ).logits
        probs = nnF.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
    
    return np.concatenate(all_probs, axis=0)


# Load test data
test_df = pd.read_parquet(TEST_PATH)
test_df = test_df.dropna(subset=['code']).reset_index(drop=True)
test_codes = test_df['code'].tolist()
test_ids = test_df['ID'].values if 'ID' in test_df.columns else np.arange(len(test_df))

print(f"Test samples: {len(test_codes)}")
print(f"Models: {list(trained_models.keys())}")

# Run inference for each model
model_probs = {}  # tag → (N, C) probabilities

for tag, trainer_obj in trained_models.items():
    probs = get_model_probs(trainer_obj, test_codes, batch_size=BATCH_SIZE * 2)
    model_probs[tag] = probs
    print(f"  [{tag}] shape={probs.shape}, mean_conf={probs.max(axis=1).mean():.4f}")

# Cache probabilities
os.makedirs(ENSEMBLE_DIR, exist_ok=True)
for tag, probs in model_probs.items():
    np.save(f"{ENSEMBLE_DIR}/{tag}_probs.npy", probs)
np.save(f"{ENSEMBLE_DIR}/sample_ids.npy", test_ids)
print(f"\n✅ Probabilities cached → {ENSEMBLE_DIR}")

Test samples: 1000
Models: ['codebert', 'graphcodebert', 'unixcoder']


  [codebert]: 100%|██████████| 32/32 [00:16<00:00,  1.89it/s]


  [codebert] shape=(1000, 2), mean_conf=0.9680


  [graphcodebert]: 100%|██████████| 32/32 [00:16<00:00,  1.92it/s]


  [graphcodebert] shape=(1000, 2), mean_conf=0.9673


  [unixcoder]: 100%|██████████| 32/32 [00:16<00:00,  1.96it/s]

  [unixcoder] shape=(1000, 2), mean_conf=0.9844

✅ Probabilities cached → /kaggle/working/ensemble_cache


In [8]:
# ============================================================
# Cell 8: Ensemble Combination Functions
# ============================================================

def soft_vote(prob_list):
    """Average probabilities across models (equal weight)."""
    return np.mean(np.stack(prob_list, axis=0), axis=0)


def weighted_average(prob_list, weights):
    """Weighted sum of probabilities (weights normalised to sum=1)."""
    w = np.array(weights, dtype=np.float64)
    w /= w.sum()
    stacked = np.stack(prob_list, axis=0)  # (K, N, C)
    return (stacked * w[:, None, None]).sum(axis=0)


def rank_average(prob_list):
    """Rank-based fusion (robust to miscalibrated models)."""
    from scipy.stats import rankdata
    ranks = [np.apply_along_axis(rankdata, 1, p) for p in prob_list]
    avg = np.mean(np.stack(ranks, axis=0), axis=0)
    return avg / avg.sum(axis=1, keepdims=True)


def optimize_weights_on_val(prob_dict, val_labels, n_steps=21):
    """
    Grid search for optimal 3-model weights on validation labels.
    Returns: (best_weights_list, best_f1)
    """
    tags = list(prob_dict.keys())
    probs = [prob_dict[t] for t in tags]
    
    best_f1, best_w = -1, None
    
    for w0 in np.linspace(0, 1, n_steps):
        for w1 in np.linspace(0, 1 - w0, max(2, int(n_steps * (1 - w0)))):
            w2 = 1.0 - w0 - w1
            if w2 < -1e-9:
                continue
            w = [w0, w1, max(0, w2)]
            combined = weighted_average(probs, w)
            preds = combined.argmax(axis=1)
            f1 = f1_score(val_labels, preds, average='macro', zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_w = w
    
    print(f"Optimal weights: {', '.join(f'{t}={w:.3f}' for t, w in zip(tags, best_w))}")
    print(f"Optimal Macro F1: {best_f1:.4f}")
    return best_w, best_f1


print("Ensemble functions defined.")

Ensemble functions defined.


In [9]:
# ============================================================
# Cell 9: (Optional) Optimise weights on validation set
# ============================================================

# Run inference on validation set for weight optimisation
val_codes = val_df['code'].tolist()
val_labels = val_df['label'].values

val_probs = {}
print("Running val inference for weight optimisation ...")
for tag, trainer_obj in trained_models.items():
    probs = get_model_probs(trainer_obj, val_codes, batch_size=BATCH_SIZE * 2)
    val_probs[tag] = probs
    preds = probs.argmax(axis=1)
    mf1 = f1_score(val_labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(val_labels, preds)
    print(f"  [{tag}] Val Macro F1={mf1:.4f}, Acc={acc:.4f}")

# Optimise
print("\nSearching for optimal weights ...")
optimal_weights, optimal_f1 = optimize_weights_on_val(val_probs, val_labels)

# Also show soft-vote baseline
sv_probs = soft_vote(list(val_probs.values()))
sv_preds = sv_probs.argmax(axis=1)
sv_f1 = f1_score(val_labels, sv_preds, average='macro', zero_division=0)
print(f"\nSoft Vote Val Macro F1: {sv_f1:.4f}")
print(f"Weighted Avg Val Macro F1: {optimal_f1:.4f}  (Δ = {optimal_f1 - sv_f1:+.4f})")

Running val inference for weight optimisation ...


  [codebert]: 100%|██████████| 79/79 [00:37<00:00,  2.08it/s]


  [codebert] Val Macro F1=0.9804, Acc=0.9804


  [graphcodebert]: 100%|██████████| 79/79 [00:38<00:00,  2.07it/s]


  [graphcodebert] Val Macro F1=0.9808, Acc=0.9808


  [unixcoder]: 100%|██████████| 79/79 [00:37<00:00,  2.10it/s]


  [unixcoder] Val Macro F1=0.9800, Acc=0.9800

Searching for optimal weights ...
Optimal weights: codebert=0.450, graphcodebert=0.055, unixcoder=0.495
Optimal Macro F1: 0.9840

Soft Vote Val Macro F1: 0.9816
Weighted Avg Val Macro F1: 0.9840  (Δ = +0.0024)


In [10]:
# ============================================================
# Cell 10: Run Ensemble on Test Set
# ============================================================

tags = list(model_probs.keys())
prob_list = [model_probs[t] for t in tags]

# Choose strategy
if ENSEMBLE_STRATEGY == "soft_vote":
    print("Strategy: Soft Voting (equal weights)")
    combined_probs = soft_vote(prob_list)

elif ENSEMBLE_STRATEGY == "weighted_avg":
    weights = ENSEMBLE_WEIGHTS or optimal_weights
    w_str = ', '.join(f'{t}={w:.3f}' for t, w in zip(tags, weights))
    print(f"Strategy: Weighted Averaging ({w_str})")
    combined_probs = weighted_average(prob_list, weights)

elif ENSEMBLE_STRATEGY == "rank_avg":
    print("Strategy: Rank Averaging")
    combined_probs = rank_average(prob_list)

# Generate predictions
ensemble_preds = combined_probs.argmax(axis=1)
confidence = combined_probs.max(axis=1)

# Agreement analysis
per_model_preds = [p.argmax(axis=1) for p in prob_list]
all_agree = np.all(
    np.stack(per_model_preds, axis=0) == per_model_preds[0][None, :],
    axis=0,
)

print(f"\nEnsemble Results:")
print(f"  Samples:     {len(ensemble_preds)}")
print(f"  Agreement:   {all_agree.mean():.2%} ({all_agree.sum()}/{len(ensemble_preds)})")
print(f"  Confidence:  mean={confidence.mean():.4f}, min={confidence.min():.4f}")
print(f"  Prediction distribution: {pd.Series(ensemble_preds).value_counts().sort_index().to_dict()}")

Strategy: Weighted Averaging (codebert=0.450, graphcodebert=0.055, unixcoder=0.495)

Ensemble Results:
  Samples:     1000
  Agreement:   71.50% (715/1000)
  Confidence:  mean=0.8800, min=0.5000
  Prediction distribution: {0: 220, 1: 780}


In [11]:
# ============================================================
# Cell 11: Create Submission CSV
# ============================================================

submission = pd.DataFrame({
    'ID': test_ids,
    'prediction': ensemble_preds.astype(int),
})

submission_path = f"{OUTPUT_BASE}/ensemble_submission_task{TASK}.csv"
submission.to_csv(submission_path, index=False)
print(f"✅ Submission saved → {submission_path}")
print(f"   {len(submission)} rows")
print(f"\nSample:")
submission.head(10)

✅ Submission saved → /kaggle/working/ensemble_submission_taskA.csv
   1000 rows

Sample:


,ID,prediction
0,0,1
1,1,0
2,2,1
3,3,1
4,4,1
5,5,1
6,6,1
7,7,1
8,8,1
9,9,1


In [12]:
# ============================================================
# Cell 12: Evaluate on Test Set (if labels available)
# ============================================================

SEEN_LANGS   = {'c++', 'cpp', 'python', 'java'}
UNSEEN_LANGS = {'go', 'php', 'c#', 'csharp', 'c', 'javascript', 'js'}
SEEN_DOMAINS   = {'algorithmic'}
UNSEEN_DOMAINS = {'research', 'production'}


def evaluate_ensemble_results(test_df, ensemble_preds, per_model_preds_dict, task="A"):
    """Full evaluation with per-category breakdown."""
    if 'label' not in test_df.columns:
        print("No 'label' column in test data — skipping evaluation.")
        return
    
    y_true = test_df['label'].values
    
    print("\n" + "=" * 70)
    print(f"  ENSEMBLE EVALUATION — Task {task}")
    print("=" * 70)
    
    # Per-model results
    print("\n  Individual Model Performance:")
    print("  " + "-" * 55)
    for tag, preds in per_model_preds_dict.items():
        mf1 = f1_score(y_true, preds, average='macro', zero_division=0)
        acc = accuracy_score(y_true, preds)
        print(f"  {tag:20s}  Macro F1={mf1:.4f}  Acc={acc:.4f}")
    
    # Ensemble results
    ens_f1 = f1_score(y_true, ensemble_preds, average='macro', zero_division=0)
    ens_acc = accuracy_score(y_true, ensemble_preds)
    print(f"  {'ENSEMBLE':20s}  Macro F1={ens_f1:.4f}  Acc={ens_acc:.4f}  ← 🏆")
    print("  " + "-" * 55)
    
    # Detailed report
    label_map = {
        'A': ['human', 'machine'],
        'B': ['human', 'deepseek', 'qwen', '01-ai', 'bigcode', 'gemma',
              'phi', 'meta-llama', 'ibm-granite', 'mistral', 'openai'],
    }
    names = label_map.get(task, None)
    if names:
        names = names[:len(set(y_true))]
    
    print("\n  Classification Report:")
    print(classification_report(y_true, ensemble_preds, target_names=names, zero_division=0))
    
    # Per-category breakdown (Task A)
    lang_col = next((c for c in test_df.columns if c.lower() in ('language', 'lang')), None)
    domain_col = next((c for c in test_df.columns if c.lower() in ('domain', 'task_type', 'category')), None)
    
    if lang_col and domain_col:
        norm = lambda v: str(v).strip().lower()
        df_eval = test_df.copy()
        df_eval['prediction'] = ensemble_preds
        df_eval['_l'] = df_eval[lang_col].apply(norm)
        df_eval['_d'] = df_eval[domain_col].apply(norm)
        
        settings = [
            ("(i)   Seen Lang & Seen Domain",    SEEN_LANGS,   SEEN_DOMAINS),
            ("(ii)  Unseen Lang & Seen Domain",   UNSEEN_LANGS, SEEN_DOMAINS),
            ("(iii) Seen Lang & Unseen Domain",   SEEN_LANGS,   UNSEEN_DOMAINS),
            ("(iv)  Unseen Lang & Unseen Domain", UNSEEN_LANGS, UNSEEN_DOMAINS),
        ]
        
        print("\n  Per-Category Breakdown:")
        for name, langs, domains in settings:
            mask = df_eval['_l'].isin(langs) & df_eval['_d'].isin(domains)
            sub = df_eval[mask]
            if len(sub) == 0:
                print(f"    {name}:  ** no samples **")
                continue
            mf1 = f1_score(sub['label'], sub['prediction'], average='macro', zero_division=0)
            acc = accuracy_score(sub['label'], sub['prediction'])
            print(f"    {name}  (n={len(sub)})  Macro F1={mf1:.4f}  Acc={acc:.4f}")
    
    print("=" * 70)


# Run evaluation
per_model_pred_dict = {tag: model_probs[tag].argmax(axis=1) for tag in tags}
evaluate_ensemble_results(test_df, ensemble_preds, per_model_pred_dict, task=TASK)


  ENSEMBLE EVALUATION — Task A

  Individual Model Performance:
  -------------------------------------------------------
  codebert              Macro F1=0.4522  Acc=0.4610
  graphcodebert         Macro F1=0.4202  Acc=0.4210
  unixcoder             Macro F1=0.3187  Acc=0.3200
  ENSEMBLE              Macro F1=0.3710  Acc=0.3710  ← 🏆
  -------------------------------------------------------

  Classification Report:
              precision    recall  f1-score   support

       human       0.84      0.24      0.37       777
     machine       0.24      0.84      0.37       223

    accuracy                           0.37      1000
   macro avg       0.54      0.54      0.37      1000
weighted avg       0.70      0.37      0.37      1000



In [13]:
# ============================================================
# Cell 13: Compare All Strategies Side-by-Side
# ============================================================

if 'label' in test_df.columns:
    y_true = test_df['label'].values
    
    print("\n" + "=" * 70)
    print("  STRATEGY COMPARISON")
    print("=" * 70)
    
    strategies = [
        ("Soft Vote",       soft_vote(prob_list)),
        ("Weighted Avg",    weighted_average(prob_list, optimal_weights)),
        ("Rank Average",    rank_average(prob_list)),
    ]
    
    for name, probs in strategies:
        preds = probs.argmax(axis=1)
        mf1 = f1_score(y_true, preds, average='macro', zero_division=0)
        acc = accuracy_score(y_true, preds)
        conf = probs.max(axis=1).mean()
        print(f"  {name:20s}  Macro F1={mf1:.4f}  Acc={acc:.4f}  Conf={conf:.4f}")
    
    print("=" * 70)
else:
    print("No labels available for strategy comparison.")


  STRATEGY COMPARISON
  Soft Vote             Macro F1=0.4063  Acc=0.4070  Conf=0.8926
  Weighted Avg          Macro F1=0.3710  Acc=0.3710  Conf=0.8800
  Rank Average          Macro F1=0.4101  Acc=0.4110  Conf=0.6350


In [14]:
# ============================================================
# Cell 14: Save Ensemble Metadata & Clean Up
# ============================================================

# Save metadata
meta = {
    "task": TASK,
    "strategy": ENSEMBLE_STRATEGY,
    "models": tags,
    "weights": [float(w) for w in (ENSEMBLE_WEIGHTS or optimal_weights)],
    "n_samples": len(ensemble_preds),
    "n_classes": int(combined_probs.shape[1]),
    "agreement_rate": float(all_agree.mean()),
    "mean_confidence": float(confidence.mean()),
}

meta_path = f"{OUTPUT_BASE}/ensemble_meta_task{TASK}.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"Metadata saved → {meta_path}")
print(json.dumps(meta, indent=2))

# Clean up GPU memory
for tag in list(trained_models.keys()):
    del trained_models[tag].model
trained_models.clear()
gc.collect()
torch.cuda.empty_cache()
print("\n✅ GPU memory freed.")

Metadata saved → /kaggle/working/ensemble_meta_taskA.json
{
  "task": "A",
  "strategy": "weighted_avg",
  "models": [
    "codebert",
    "graphcodebert",
    "unixcoder"
  ],
  "weights": [
    0.45,
    0.05500000000000001,
    0.49500000000000005
  ],
  "n_samples": 1000,
  "n_classes": 2,
  "agreement_rate": 0.715,
  "mean_confidence": 0.880030303718742
}

✅ GPU memory freed.
